In [5]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd   
MAX_LEN = 75
data_files = {
    "train": "../data/train_cleaned_dataset.csv",
    "test": "../data/test_cleaned_dataset.csv",
}

train_df = pd.read_csv("../data/train_cleaned_dataset.csv") 
train_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2253210 entries, 0 to 2253209
Data columns (total 2 columns):
 #   Column  Dtype 
---  ------  ----- 
 0   en      object
 1   vi      object
dtypes: object(2)
memory usage: 34.4+ MB


In [7]:
bilingual_df, viet_only_df = train_df[:10], train_df[:20]
test_data = pd.read_csv("../data/test_cleaned_dataset.csv")
test_data = test_data[:]
bilingual_df.info()
viet_only_df.info()
test_data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   en      10 non-null     object
 1   vi      10 non-null     object
dtypes: object(2)
memory usage: 292.0+ bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   en      20 non-null     object
 1   vi      20 non-null     object
dtypes: object(2)
memory usage: 452.0+ bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 278175 entries, 0 to 278174
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   en      278175 non-null  object
 1   vi      278175 non-null  object
dtypes: object(2)
memory usage: 4.2+ MB


In [8]:
bilingual_df.to_csv("../datatest/bilingual_lor.csv",index=False)
viet_only_df.to_csv("../datatest/vie_lor.csv",index=False)
# test_data.to_csv("../datatest/test_cleaned_dataset.csv",index=False)

In [1]:
import pandas as pd

In [2]:
bilingual_df = pd.read_csv("../datatest/bilingual_cleaned_dataset.csv")
viet_only_df = pd.read_csv("../datatest/vie_cleaned_dataset.csv")
test_data = pd.read_csv("../datatest/test_cleaned_dataset.csv")
bilingual_df.info()
viet_only_df.info() 


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 2 columns):
 #   Column  Non-Null Count    Dtype 
---  ------  --------------    ----- 
 0   en      1000000 non-null  object
 1   vi      1000000 non-null  object
dtypes: object(2)
memory usage: 15.3+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2253210 entries, 0 to 2253209
Data columns (total 2 columns):
 #   Column  Dtype 
---  ------  ----- 
 0   en      object
 1   vi      object
dtypes: object(2)
memory usage: 34.4+ MB


In [3]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 278175 entries, 0 to 278174
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   en      278175 non-null  object
 1   vi      278175 non-null  object
dtypes: object(2)
memory usage: 4.2+ MB


In [ ]:
import pandas as pd
from nltk.translate import AlignedSent
from nltk.translate.ibm1 import IBMModel1
from nltk.lm import MLE
from nltk.lm.preprocessing import padded_everygram_pipeline
from collections import defaultdict, Counter
import math
import os
from tqdm import tqdm
import pickle
import random
import gc
# from multiprocessing import Pool, cpu_count

# Configuration constants
BEAM_SIZE = 3 
MAX_PHRASE_LENGTH = 7  
LM_ORDER = 3  
ALPHA = 0.7
BETA = 0.3
BATCH_SIZE = 1000  # For processing data in batches
MIN_PHRASE_COUNT = 3  # Increased threshold to reduce phrase table size

class MemoryOptimizedLanguageModel:
    """Memory-optimized Language Model"""
    def __init__(self, order=LM_ORDER):
        self.order = order
        self.lm = None
        self.vocab_size = 0
    
    def preprocess(self, text):
        """Tokenize Vietnamese words"""
        return text.lower().split()
    
    def train(self, vietnamese_sentences, max_sentences=200000):
        """Training Language Model with memory optimization"""
        print(f"Training Language Model on {min(len(vietnamese_sentences), max_sentences)} sentences...")
        
        # Limit training data for LM to reduce memory
        if len(vietnamese_sentences) > max_sentences:
            print(f"Sampling {max_sentences} sentences from {len(vietnamese_sentences)} for LM training")
            vietnamese_sentences = random.sample(vietnamese_sentences, max_sentences)
        
        # Process in batches to reduce memory usage
        all_tokens = []
        batch_size = 10000
        
        for i in range(0, len(vietnamese_sentences), batch_size):
            batch = vietnamese_sentences[i:i+batch_size]
            batch_tokens = [self.preprocess(sent) for sent in batch]
            all_tokens.extend(batch_tokens)
            
            # Force garbage collection
            if i % (batch_size * 5) == 0:
                gc.collect()
        
        # Build vocabulary with size limit
        vocab = set()
        for tokens in all_tokens:
            vocab.update(tokens)
        
        # Limit vocabulary size to most frequent words
        if len(vocab) > 50000:
            word_freq = Counter()
            for tokens in all_tokens:
                word_freq.update(tokens)
            
            # Keep only top 50k words
            most_common = word_freq.most_common(50000)
            vocab = set(word for word, _ in most_common)
            print(f"Limited vocabulary to {len(vocab)} most frequent words")
        
        self.vocab_size = len(vocab)
        
        # Filter sentences to contain only vocabulary words
        filtered_sentences = []
        for tokens in all_tokens:
            filtered_tokens = [token for token in tokens if token in vocab]
            if filtered_tokens:  # Only add non-empty sentences
                filtered_sentences.append(filtered_tokens)
        
        # Clear original data
        del all_tokens
        gc.collect()
        
        # Train N-gram model
        train_data, padded_sents = padded_everygram_pipeline(self.order, filtered_sentences)
        self.lm = MLE(self.order)
        self.lm.fit(train_data, padded_sents)
        
        # Clear training data
        del filtered_sentences, train_data, padded_sents
        gc.collect()
        
        return {"vocab_size": self.vocab_size, "ngram_order": self.order}
    
    def get_probability(self, tokens):
        """Calculate probability P(V) for a vietnamese tokens sequence"""
        if not tokens or not self.lm:
            return 0.0
        
        start_tokens = ['<s>'] * (self.order - 1)
        tokens = start_tokens + tokens
        log_prob = 0.0
        
        for i in range(self.order - 1, len(tokens)):
            context = tokens[max(0, i - self.order + 1):i]
            word = tokens[i]
            prob = self.lm.score(word, context) or 1e-10
            log_prob += math.log(prob)
        
        return log_prob

class MemoryOptimizedTranslationModel:
    """Memory-optimized Translation Model"""
    def __init__(self, max_phrase_length=MAX_PHRASE_LENGTH):
        self.max_phrase_length = max_phrase_length
        self.phrase_table = {}
        self.word_alignments = []
        
    def preprocess(self, text, lang):
        """Preprocess text for both languages"""
        return text.lower().split()
    
    def load_bilingual_data_batch(self, file_path, batch_size=BATCH_SIZE):
        """Load bilingual data in batches to reduce memory usage"""
        print(f"Loading bilingual data from {file_path} in batches")
        
        try:
            df = pd.read_csv(file_path)
        except FileNotFoundError:
            file_path = '/kaggle/input/general-data/bilingual_cleaned_dataset.csv'
            df = pd.read_csv(file_path)
        
        total_rows = len(df)
        print(f"Total rows: {total_rows}")
        
        # Process in batches
        for start_idx in range(0, total_rows, batch_size):
            end_idx = min(start_idx + batch_size, total_rows)
            batch_df = df.iloc[start_idx:end_idx]
            
            aligned_sentences = []
            for _, row in batch_df.iterrows():
                eng_tokens = self.preprocess(row['en'], 'eng')
                vie_tokens = self.preprocess(row['vi'], 'vie')
                
                # Filter out very long sentences to save memory
                if len(eng_tokens) <= 50 and len(vie_tokens) <= 50:
                    aligned_sentences.append(AlignedSent(eng_tokens, vie_tokens))
            
            yield aligned_sentences
            
            # Clean up batch
            del batch_df, aligned_sentences
            gc.collect()
    
    def train_ibm_model_incremental(self, file_path, iterations=5):
        """Train IBM Model 1 incrementally to reduce memory usage"""
        print(f"Training IBM Model 1 incrementally with {iterations} iterations...")
        
        # First pass: collect vocabulary and create aligned sentences
        all_aligned_sentences = []
        eng_vocab = set()
        vie_vocab = set()
        
        for batch in self.load_bilingual_data_batch(file_path):
            for sent_pair in batch:
                eng_vocab.update(sent_pair.words)
                vie_vocab.update(sent_pair.mots)
                all_aligned_sentences.append(sent_pair)
            
            # Limit total sentences to prevent memory issues
            if len(all_aligned_sentences) >= 300000:  # Reduced from 500k
                print(f"Limited training to {len(all_aligned_sentences)} sentences")
                break
        
        print(f"Training on {len(all_aligned_sentences)} aligned sentences")
        print(f"English vocab: {len(eng_vocab)}, Vietnamese vocab: {len(vie_vocab)}")
        
        # Train IBM Model with reduced iterations
        ibm_model = IBMModel1(all_aligned_sentences, iterations)
        
        # Extract alignments with memory optimization
        self.word_alignments = self._extract_alignments_memory_efficient(all_aligned_sentences, ibm_model)
        
        # Clean up
        del ibm_model
        gc.collect()
        
        return all_aligned_sentences
    
    def _extract_alignments_memory_efficient(self, aligned_sentences, ibm_model):
        """Memory-efficient alignment extraction"""
        alignments = []
        
        # Process in smaller batches
        batch_size = 5000
        for i in range(0, len(aligned_sentences), batch_size):
            batch_alignments = []
            batch_sentences = aligned_sentences[i:i+batch_size]
            
            for sent_pair in batch_sentences:
                eng_tokens = sent_pair.words
                vie_tokens = sent_pair.mots
                
                # Only keep high-probability alignments
                alignment = []
                for eng_i, eng_word in enumerate(eng_tokens):
                    best_prob = 0
                    best_vie_i = -1
                    
                    for vie_i, vie_word in enumerate(vie_tokens):
                        prob = ibm_model.translation_table.get(eng_word, {}).get(vie_word, 0)
                        if prob > best_prob:
                            best_prob = prob
                            best_vie_i = vie_i
                    
                    # Only keep alignments above threshold
                    if best_prob > 0.01:  # Increased threshold
                        alignment.append((eng_i, best_vie_i))
                
                batch_alignments.append(alignment)
            
            alignments.extend(batch_alignments)
            
            # Periodic cleanup
            if i % (batch_size * 10) == 0:
                gc.collect()
        
        return alignments
    
    def extract_phrases_memory_efficient(self, aligned_sentences):
        """Memory-efficient phrase extraction"""
        print("Extracting phrase pairs with memory optimization...")
        
        # Use smaller data structures
        phrase_counts = defaultdict(lambda: defaultdict(int))
        
        # Process in batches
        batch_size = 5000
        for i in range(0, len(aligned_sentences), batch_size):
            batch_sentences = aligned_sentences[i:i+batch_size]
            batch_alignments = self.word_alignments[i:i+batch_size]
            
            for sent_pair, alignments in zip(batch_sentences, batch_alignments):
                if not alignments:  # Skip sentences with no alignments
                    continue
                    
                eng_tokens = sent_pair.words
                vie_tokens = sent_pair.mots
                alignment_set = set(alignments)
                
                # Extract word-level translations first
                for eng_i, vie_i in alignments:
                    if eng_i < len(eng_tokens) and vie_i < len(vie_tokens):
                        eng_word = eng_tokens[eng_i]
                        vie_word = vie_tokens[vie_i]
                        phrase_counts[eng_word][vie_word] += 1
                
                # Extract short phrases only (max length 3 to save memory)
                max_len = min(3, self.max_phrase_length)
                consistent_phrases = self._extract_consistent_phrases(
                    eng_tokens, vie_tokens, alignment_set, max_len
                )
                
                for eng_phrase, vie_phrase in consistent_phrases:
                    phrase_counts[eng_phrase][vie_phrase] += 1
            
            # Periodic cleanup
            if i % (batch_size * 5) == 0:
                gc.collect()
                print(f"Processed {i+batch_size} sentences...")
        
        # Calculate probabilities with higher threshold
        self.phrase_table = {}
        for eng_phrase, vie_phrases in phrase_counts.items():
            total_count = sum(vie_phrases.values())
            if total_count >= MIN_PHRASE_COUNT:  # Higher threshold
                # Keep only top 3 translations per phrase to save memory
                sorted_phrases = sorted(vie_phrases.items(), key=lambda x: x[1], reverse=True)[:3]
                
                filtered_phrases = {}
                for vie_phrase, count in sorted_phrases:
                    if count >= MIN_PHRASE_COUNT:
                        filtered_phrases[vie_phrase] = count / total_count
                
                if filtered_phrases:
                    self.phrase_table[eng_phrase] = filtered_phrases
        
        print(f"Extracted {len(self.phrase_table)} phrase pairs (filtered)")
        return self.phrase_table
    
    def _extract_consistent_phrases(self, eng_tokens, vie_tokens, alignments, max_length):
        """Extract consistent phrase pairs with length limit"""
        consistent_phrases = []
        eng_len = len(eng_tokens)
        vie_len = len(vie_tokens)
        
        # Limit phrase extraction to reduce memory
        for e_start in range(eng_len):
            for e_end in range(e_start, min(eng_len, e_start + max_length)):
                vie_positions = set()
                for e_pos in range(e_start, e_end + 1):
                    for (eng_idx, vie_idx) in alignments:
                        if eng_idx == e_pos:
                            vie_positions.add(vie_idx)
                
                if not vie_positions:
                    continue
                
                v_start, v_end = min(vie_positions), max(vie_positions)
                
                if v_end - v_start + 1 <= max_length:
                    if self._is_consistent_phrase_pair(e_start, e_end, v_start, v_end, alignments):
                        eng_phrase = ' '.join(eng_tokens[e_start:e_end+1])
                        vie_phrase = ' '.join(vie_tokens[v_start:v_end+1])
                        consistent_phrases.append((eng_phrase, vie_phrase))
        
        return consistent_phrases
    
    def _is_consistent_phrase_pair(self, e_start, e_end, v_start, v_end, alignments):
        """Check if a phrase pair is consistent"""
        for (eng_idx, vie_idx) in alignments:
            if (e_start <= eng_idx <= e_end) and not (v_start <= vie_idx <= v_end):
                return False
            if (v_start <= vie_idx <= v_end) and not (e_start <= eng_idx <= e_end):
                return False
        return True

class MemoryOptimizedDecoder:
    """Memory-optimized decoder"""
    def __init__(self, phrase_table, language_model, beam_size=BEAM_SIZE):
        self.phrase_table = phrase_table
        self.lm = language_model
        self.beam_size = beam_size
        
    def translate(self, sentence):
        """Translate sentence with memory optimization"""
        tokens = sentence.lower().split()
        if not tokens:
            return ""
        
        # Use greedy search instead of beam search for memory efficiency
        return self._greedy_translate(tokens)
    
    def _greedy_translate(self, tokens):
        """Greedy translation to save memory"""
        translation = []
        i = 0
        
        while i < len(tokens):
            best_phrase_len = 1
            best_translation = tokens[i]  # fallback
            
            # Try phrases of different lengths
            for phrase_len in range(min(3, len(tokens) - i), 0, -1):  # Max length 3
                eng_phrase = ' '.join(tokens[i:i+phrase_len])
                
                if eng_phrase in self.phrase_table:
                    # Get best translation
                    vie_translations = self.phrase_table[eng_phrase]
                    if vie_translations:
                        best_vie_phrase = max(vie_translations.items(), key=lambda x: x[1])
                        best_translation = best_vie_phrase[0]
                        best_phrase_len = phrase_len
                        break
            
            translation.append(best_translation)
            i += best_phrase_len
        
        return ' '.join(translation)

class Hypothesis:
    """Lightweight hypothesis class"""
    def __init__(self, translation, coverage, score, last_phrase_end):
        self.translation = translation
        self.coverage = coverage
        self.score = score
        self.last_phrase_end = last_phrase_end

class MemoryOptimizedSMT:
    """Memory-optimized SMT system"""
    def __init__(self, data_path="../datatest/"):
        self.data_path = data_path
        self.lm = MemoryOptimizedLanguageModel(order=LM_ORDER)
        self.tm = MemoryOptimizedTranslationModel(max_phrase_length=MAX_PHRASE_LENGTH)
        self.decoder = None
        
    def train(self, file_path=None):
        """Train the SMT system with memory optimization"""
        if file_path is None:
            file_path = "/kaggle/input/general-data/bilingual_cleaned_dataset.csv"
        
        vie_path = "/kaggle/input/general-data/vie_cleaned_dataset.csv"
        
        # Train translation model
        print("=== Training Translation Model ===")
        aligned_sentences = self.tm.train_ibm_model_incremental(file_path, iterations=5)
        phrase_table = self.tm.extract_phrases_memory_efficient(aligned_sentences)
        
        # Clear aligned sentences to free memory
        del aligned_sentences
        gc.collect()
        
        # Train language model
        print("\n=== Training Language Model ===")
        vie_df = pd.read_csv(vie_path)
        vietnamese_sentences = vie_df['vi'].tolist()
        del vie_df  # Free memory
        gc.collect()
        
        lm_stats = self.lm.train(vietnamese_sentences, max_sentences=50000)  # Limit LM training data
        del vietnamese_sentences  # Free memory
        gc.collect()
        
        # Initialize decoder
        self.decoder = MemoryOptimizedDecoder(phrase_table, self.lm, BEAM_SIZE)
        
        # Save model immediately
        self.save_model()
        
        return {
            "phrase_pairs": len(phrase_table),
            "lm_stats": lm_stats
        }
    
    def translate_sentence(self, sentence):
        """Translate a single sentence"""
        if self.decoder is None:
            raise ValueError("Model not trained or loaded.")
        return self.decoder.translate(sentence)
    
    def save_model(self, model_dir=None):
        """Save the trained model"""
        if model_dir is None:
            model_dir = "/kaggle/working/model"
        
        os.makedirs(model_dir, exist_ok=True)
        
        # Save with compression
        with open(os.path.join(model_dir, "phrase_table.pkl"), 'wb') as f:
            pickle.dump(self.tm.phrase_table, f, protocol=pickle.HIGHEST_PROTOCOL)
        with open(os.path.join(model_dir, "lm_object.pkl"), 'wb') as f:
            pickle.dump(self.lm, f, protocol=pickle.HIGHEST_PROTOCOL)
        
        print(f"Model saved to {model_dir}")
    
    def load_model(self, model_dir=None):
        """Load a pre-trained model"""
        if model_dir is None:
            model_dir = "/kaggle/working/model"
        
        with open(os.path.join(model_dir, "phrase_table.pkl"), 'rb') as f:
            phrase_table = pickle.load(f)
        with open(os.path.join(model_dir, "lm_object.pkl"), 'rb') as f:
            self.lm = pickle.load(f)
        
        self.decoder = MemoryOptimizedDecoder(phrase_table, self.lm, BEAM_SIZE)
        self.tm.phrase_table = phrase_table
        
        print(f"Model loaded from {model_dir}")
    
    def evaluate(self, test_file=None, sample_size=5):
        """Evaluate model on test set"""
        if test_file is None:
            test_file = "/kaggle/input/general-data/test_cleaned_dataset.csv"
        
        df = pd.read_csv(test_file)
        sample_size = min(sample_size, len(df))
        sample_indices = random.sample(range(len(df)), sample_size)
        
        results = []
        for idx in sample_indices:
            try:
                source = df.iloc[idx]['en']
                reference = df.iloc[idx]['vi']
                translation = self.translate_sentence(source)
                
                results.append({
                    "source": source,
                    "reference": reference,
                    "translation": translation
                })
            except Exception as e:
                print(f"Error translating sentence {idx}: {e}")
                results.append({
                    "source": df.iloc[idx]['en'],
                    "reference": df.iloc[idx]['vi'],
                    "translation": "Translation failed"
                })
        
        return results
    
    def save_predictions_batch(self, test_file=None, output_file=None, batch_size=1000):
        """Save predictions in batches to avoid memory issues"""
        if test_file is None:
            test_file = "/kaggle/input/general-data/test_cleaned_dataset.csv"
        if output_file is None:
            output_file = "/kaggle/working/predicted.csv"
        
        # Read file info
        df_info = pd.read_csv(test_file, nrows=0)  # Just get column info
        total_rows = len(pd.read_csv(test_file))
        
        print(f"Processing {total_rows} sentences in batches of {batch_size}")
        
        # Process in batches and write incrementally
        first_batch = True
        
        for start_idx in tqdm(range(0, total_rows, batch_size), desc="Processing batches"):
            end_idx = min(start_idx + batch_size, total_rows)
            
            # Read batch
            batch_df = pd.read_csv(test_file, skiprows=range(1, start_idx+1), nrows=batch_size)
            
            # Process batch
            batch_predictions = []
            for _, row in batch_df.iterrows():
                try:
                    source = row['en']
                    reference = row['vi']
                    translation = self.translate_sentence(source)
                    
                    batch_predictions.append({
                        "en": source,
                        "vi": reference,
                        "pre": translation
                    })
                except Exception as e:
                    batch_predictions.append({
                        "en": row['en'],
                        "vi": row['vi'],
                        "pre": "Translation failed"
                    })
            
            # Save batch
            batch_pred_df = pd.DataFrame(batch_predictions)
            
            if first_batch:
                batch_pred_df.to_csv(output_file, index=False)
                first_batch = False
            else:
                batch_pred_df.to_csv(output_file, mode='a', header=False, index=False)
            
            # Clean up
            del batch_df, batch_predictions, batch_pred_df
            gc.collect()
        
        print(f"Predictions saved to {output_file}")
        return output_file

def main():
    """Main function with memory optimization"""
    print("Starting Memory-Optimized SMT System...")
    smt = MemoryOptimizedSMT()
    
    model_dir = "/kaggle/working/model"
    if os.path.exists(model_dir) and os.path.isfile(os.path.join(model_dir, "phrase_table.pkl")):
        print("Loading existing model...")
        smt.load_model()
    else:
        print("Training new model...")
        stats = smt.train()
        print(f"Training complete: {stats}")
    
    # Evaluate model
    print("\nEvaluating model...")
    results = smt.evaluate(sample_size=3)  # Reduced sample size
    
    print("\nExample translations:")
    for i, result in enumerate(results):
        print(f"\nExample {i+1}:")
        print(f"English:    {result['source']}")
        print(f"Reference:  {result['reference']}")
        print(f"Translation: {result['translation']}")
    
    # Save predictions in batches
    print("\nSaving predictions in batches...")
    output_file = smt.save_predictions_batch(batch_size=500)  # Smaller batch size
    print(f"All predictions saved to: {output_file}")
    
    # Final memory cleanup
    gc.collect()
    print("Processing complete!")

if __name__ == "__main__":
    main()

In [3]:
import pandas as pd
import numpy as np
from typing import List, Dict
import os
import re
from collections import Counter
import math

def simple_bleu_score(hypothesis: str, reference: str, n: int = 4) -> float:
    """Tính BLEU score đơn giản"""
    def get_ngrams(text: str, n: int) -> List[tuple]:
        tokens = text.lower().split()
        return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    
    # Tính precision cho từng n-gram
    precisions = []
    for i in range(1, n+1):
        hyp_ngrams = get_ngrams(hypothesis, i)
        ref_ngrams = get_ngrams(reference, i)
        
        if len(hyp_ngrams) == 0:
            precisions.append(0.0)
            continue
            
        hyp_counter = Counter(hyp_ngrams)
        ref_counter = Counter(ref_ngrams)
        
        overlap = sum(min(hyp_counter[ngram], ref_counter[ngram]) for ngram in hyp_counter)
        precision = overlap / len(hyp_ngrams)
        precisions.append(precision)
    
    # Brevity penalty
    hyp_len = len(hypothesis.split())
    ref_len = len(reference.split())
    if hyp_len == 0:
        return 0.0
    
    bp = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len/hyp_len)
    
    # Geometric mean
    if all(p > 0 for p in precisions):
        bleu = bp * math.exp(sum(math.log(p) for p in precisions) / len(precisions))
    else:
        bleu = 0.0
    
    return bleu

def rouge_n_score(hypothesis: str, reference: str, n: int = 1) -> Dict[str, float]:
    """Tính ROUGE-N score"""
    def get_ngrams(text: str, n: int) -> List[tuple]:
        tokens = text.lower().split()
        return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    
    hyp_ngrams = get_ngrams(hypothesis, n)
    ref_ngrams = get_ngrams(reference, n)
    
    if len(ref_ngrams) == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    
    hyp_counter = Counter(hyp_ngrams)
    ref_counter = Counter(ref_ngrams)
    
    overlap = sum(min(hyp_counter[ngram], ref_counter[ngram]) for ngram in hyp_counter)
    
    precision = overlap / len(hyp_ngrams) if len(hyp_ngrams) > 0 else 0.0
    recall = overlap / len(ref_ngrams)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {"precision": precision, "recall": recall, "f1": f1}

def rouge_l_score(hypothesis: str, reference: str) -> Dict[str, float]:
    """Tính ROUGE-L score (Longest Common Subsequence)"""
    def lcs_length(x: List[str], y: List[str]) -> int:
        m, n = len(x), len(y)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if x[i-1] == y[j-1]:
                    dp[i][j] = dp[i-1][j-1] + 1
                else:
                    dp[i][j] = max(dp[i-1][j], dp[i][j-1])
        
        return dp[m][n]
    
    hyp_tokens = hypothesis.lower().split()
    ref_tokens = reference.lower().split()
    
    if len(ref_tokens) == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    
    lcs_len = lcs_length(hyp_tokens, ref_tokens)
    
    precision = lcs_len / len(hyp_tokens) if len(hyp_tokens) > 0 else 0.0
    recall = lcs_len / len(ref_tokens)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {"precision": precision, "recall": recall, "f1": f1}

def character_level_metrics(hypothesis: str, reference: str) -> Dict[str, float]:
    """Tính metrics ở mức ký tự"""
    # Loại bỏ khoảng trắng và chuyển về lowercase
    hyp_chars = set(hypothesis.lower().replace(" ", ""))
    ref_chars = set(reference.lower().replace(" ", ""))
    
    if len(ref_chars) == 0:
        return {"char_precision": 0.0, "char_recall": 0.0, "char_f1": 0.0}
    
    overlap = len(hyp_chars & ref_chars)
    
    precision = overlap / len(hyp_chars) if len(hyp_chars) > 0 else 0.0
    recall = overlap / len(ref_chars)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {"char_precision": precision, "char_recall": recall, "char_f1": f1}

def edit_distance(s1: str, s2: str) -> int:
    """Tính Levenshtein distance"""
    if len(s1) < len(s2):
        return edit_distance(s2, s1)
    
    if len(s2) == 0:
        return len(s1)
    
    previous_row = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    
    return previous_row[-1]

def compute_simple_metrics(hypotheses: List[str], references: List[str]) -> Dict[str, float]:
    """Tính toán các metrics đơn giản không cần thư viện phức tạp"""
    
    if len(hypotheses) != len(references):
        raise ValueError("Số lượng hypotheses và references phải bằng nhau")
    
    metrics = {}
    all_scores = {
        'bleu': [],
        'rouge1_f1': [],
        'rouge2_f1': [],
        'rougel_f1': [],
        'char_f1': [],
        'edit_distance': [],
        'length_ratio': []
    }
    
    print(f"Đang tính toán metrics cho {len(hypotheses)} mẫu...")
    
    for i, (hyp, ref) in enumerate(zip(hypotheses, references)):
        if i % 100 == 0:
            print(f"Đã xử lý: {i}/{len(hypotheses)}")
        
        # BLEU Score
        bleu = simple_bleu_score(hyp, ref)
        all_scores['bleu'].append(bleu)
        
        # ROUGE Scores
        rouge1 = rouge_n_score(hyp, ref, 1)
        rouge2 = rouge_n_score(hyp, ref, 2)
        rougel = rouge_l_score(hyp, ref)
        
        all_scores['rouge1_f1'].append(rouge1['f1'])
        all_scores['rouge2_f1'].append(rouge2['f1'])
        all_scores['rougel_f1'].append(rougel['f1'])
        
        # Character-level metrics
        char_metrics = character_level_metrics(hyp, ref)
        all_scores['char_f1'].append(char_metrics['char_f1'])
        
        # Edit distance (normalized)
        edit_dist = edit_distance(hyp.lower(), ref.lower())
        max_len = max(len(hyp), len(ref))
        normalized_edit_dist = 1 - (edit_dist / max_len) if max_len > 0 else 0.0
        all_scores['edit_distance'].append(normalized_edit_dist)
        
        # Length ratio
        hyp_len = len(hyp.split())
        ref_len = len(ref.split())
        length_ratio = min(hyp_len, ref_len) / max(hyp_len, ref_len) if max(hyp_len, ref_len) > 0 else 0.0
        all_scores['length_ratio'].append(length_ratio)
    
    # Tính trung bình
    metrics["BLEU"] = np.mean(all_scores['bleu'])
    metrics["ROUGE-1"] = np.mean(all_scores['rouge1_f1'])
    metrics["ROUGE-2"] = np.mean(all_scores['rouge2_f1'])
    metrics["ROUGE-L"] = np.mean(all_scores['rougel_f1'])
    metrics["Character F1"] = np.mean(all_scores['char_f1'])
    metrics["Edit Distance Similarity"] = np.mean(all_scores['edit_distance'])
    metrics["Length Ratio"] = np.mean(all_scores['length_ratio'])
    
    return metrics

def evaluate_translation_results(csv_path: str):
    """
    Đọc file CSV và tính toán metrics giữa cột 'pre' và 'vi'
    """
    try:
        # Đọc file CSV
        print(f"Đang đọc file: {csv_path}")
        df = pd.read_csv(csv_path)
        
        # Kiểm tra các cột cần thiết
        required_columns = ['pre', 'vi']
        missing_columns = [col for col in required_columns if col not in df.columns]
        
        if missing_columns:
            raise ValueError(f"Thiếu các cột: {missing_columns}. Các cột hiện có: {list(df.columns)}")
        
        # Lọc bỏ các dòng có giá trị NaN
        df_clean = df.dropna(subset=['pre', 'vi'])
        
        if len(df_clean) == 0:
            raise ValueError("Không có dữ liệu hợp lệ sau khi loại bỏ các dòng NaN")
        
        print(f"Tổng số mẫu: {len(df)}")
        print(f"Số mẫu hợp lệ: {len(df_clean)}")
        
        # Chuẩn bị dữ liệu
        hypotheses = df_clean['pre'].astype(str).tolist()  # Dự đoán (predictions)
        references = df_clean['vi'].astype(str).tolist()   # Tham chiếu (ground truth)
        
        print("\n" + "="*50)
        print("ĐANG TÍNH TOÁN METRICS...")
        print("="*50)
        
        # Tính toán metrics
        metrics = compute_simple_metrics(hypotheses, references)
        
        # Hiển thị kết quả
        print("\n" + "="*50)
        print("KẾT QUẢ ĐÁNH GIÁ")
        print("="*50)
        
        for metric_name, score in metrics.items():
            print(f"{metric_name:<25}: {score:.4f}")
        
        # Thống kê thêm
        print("\n" + "="*30)
        print("THỐNG KÊ BỔ SUNG")
        print("="*30)
        
        avg_hyp_len = np.mean([len(h.split()) for h in hypotheses])
        avg_ref_len = np.mean([len(r.split()) for r in references])
        
        print(f"Độ dài trung bình (dự đoán): {avg_hyp_len:.2f} từ")
        print(f"Độ dài trung bình (tham chiếu): {avg_ref_len:.2f} từ")
        print(f"Tỷ lệ độ dài: {avg_hyp_len/avg_ref_len:.2f}")
        
        # Lưu kết quả vào file
        results_file = csv_path.replace('.csv', '_evaluation_results.txt')
        with open(results_file, 'w', encoding='utf-8') as f:
            f.write("TRANSLATION EVALUATION RESULTS\n")
            f.write("="*50 + "\n")
            f.write(f"File: {csv_path}\n")
            f.write(f"Total samples: {len(df)}\n")
            f.write(f"Valid samples: {len(df_clean)}\n\n")
            
            f.write("METRICS:\n")
            for metric_name, score in metrics.items():
                f.write(f"{metric_name:<25}: {score:.4f}\n")
            
            f.write(f"\nSTATISTICS:\n")
            f.write(f"Avg hypothesis length: {avg_hyp_len:.2f} words\n")
            f.write(f"Avg reference length: {avg_ref_len:.2f} words\n")
            f.write(f"Length ratio: {avg_hyp_len/avg_ref_len:.2f}\n")
        
        print(f"\nKết quả đã được lưu vào: {results_file}")
        
        return metrics
        
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file {csv_path}")
        return None
    except Exception as e:
        print(f"Lỗi: {str(e)}")
        return None

def main():
    # Đường dẫn tới file predicted.csv
    csv_path = "../datatest/predicted.csv"
    
    # Kiểm tra file có tồn tại không
    if not os.path.exists(csv_path):
        print(f"File không tồn tại: {csv_path}")
        print("Vui lòng kiểm tra đường dẫn và thử lại.")
        return
    
    print("ĐÁNH GIÁ DỊCH THUẬT - PHIÊN BẢN LOCAL")
    print("="*50)
    print("Sử dụng các metrics cơ bản, không cần thư viện phức tạp")
    print("="*50)
    
    # Chạy đánh giá
    results = evaluate_translation_results(csv_path)
    
    if results:
        print("\n" + "="*50)
        print("HOÀN THÀNH ĐÁNH GIÁ!")
        print("="*50)
        print("\nCác metrics được tính:")
        print("• BLEU: Độ chính xác n-gram")
        print("• ROUGE-1/2/L: Độ tương đồng unigram/bigram/longest sequence")
        print("• Character F1: Độ tương đồng ở mức ký tự")
        print("• Edit Distance: Độ tương đồng dựa trên khoảng cách chỉnh sửa")
        print("• Length Ratio: Tỷ lệ độ dài câu")
    else:
        print("Đánh giá thất bại. Vui lòng kiểm tra lại dữ liệu.")

if __name__ == "__main__":
    main()

ĐÁNH GIÁ DỊCH THUẬT - PHIÊN BẢN LOCAL
Sử dụng các metrics cơ bản, không cần thư viện phức tạp
Đang đọc file: ../datatest/predicted.csv
Tổng số mẫu: 278175
Số mẫu hợp lệ: 278175

ĐANG TÍNH TOÁN METRICS...
Đang tính toán metrics cho 278175 mẫu...
Đã xử lý: 0/278175
Đã xử lý: 100/278175
Đã xử lý: 200/278175
Đã xử lý: 300/278175
Đã xử lý: 400/278175
Đã xử lý: 500/278175
Đã xử lý: 600/278175
Đã xử lý: 700/278175
Đã xử lý: 800/278175
Đã xử lý: 900/278175
Đã xử lý: 1000/278175
Đã xử lý: 1100/278175
Đã xử lý: 1200/278175
Đã xử lý: 1300/278175
Đã xử lý: 1400/278175
Đã xử lý: 1500/278175
Đã xử lý: 1600/278175
Đã xử lý: 1700/278175
Đã xử lý: 1800/278175
Đã xử lý: 1900/278175
Đã xử lý: 2000/278175
Đã xử lý: 2100/278175
Đã xử lý: 2200/278175
Đã xử lý: 2300/278175
Đã xử lý: 2400/278175
Đã xử lý: 2500/278175
Đã xử lý: 2600/278175
Đã xử lý: 2700/278175
Đã xử lý: 2800/278175
Đã xử lý: 2900/278175
Đã xử lý: 3000/278175
Đã xử lý: 3100/278175
Đã xử lý: 3200/278175
Đã xử lý: 3300/278175
Đã xử lý: 3400/2781